# Track A Supervised MVTec Classification

## Purpose

This notebook is a governed execution and reporting wrapper for Track A supervised binary classification on the MVTec supervised benchmark split.

## Scope

This notebook does:

- Resolve runtime, repository, config, manifest, and artifact paths.
- Inspect governed configuration and split metadata.
- Use the existing project data loader for a lightweight data smoke inspection.
- Provide a prepared, opt-in cell to call the existing training pipeline.
- Read TrainingResult and validation evaluation artifacts after training has produced them.
- Display metrics, confusion matrix evidence, sample inputs, and a final run summary.

This notebook does not:

- Implement a training loop.
- Implement metric computation.
- Generate or mutate dataset splits.
- Modify source code, configs, manifests, or artifacts.
- Tune hyperparameters or optimize model performance.
- Replace the governed source pipeline under `src/`.

> Governance warning: `mvtec_classification_supervised` is an internal supervised binary classification benchmark. It is not the official MVTec anomaly benchmark split.


## 1. Runtime Detection

Detect whether the notebook is running in Colab or locally, show runtime details, resolve the repository root, and make `src/` importable. Failure to resolve the repository or import `inspection_ai` is blocking.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import platform
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "inspection_ai").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Unable to resolve repository root containing src/inspection_ai and configs/.")

REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

try:
    import inspection_ai  # noqa: F401
    SRC_IMPORTABLE = True
except ImportError as exc:
    raise RuntimeError("src/ is not importable; cannot use governed project code.") from exc

print(f"runtime={'colab' if IN_COLAB else 'local'}")
print(f"python={platform.python_version()}")
print(f"repo_root={REPO_ROOT}")
print(f"src_importable={SRC_IMPORTABLE}")

## 2. Path And Data Source Resolution

Resolve paths in a Colab-first way, then use local repository fallback. Required config, manifest, and raw data paths are blocking.

In [ ]:
def resolve_path(path: str | Path, base: Path = REPO_ROOT) -> Path:
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate.resolve()
    return (base / candidate).resolve()


def resolve_required_path(path: str | Path, description: str) -> Path:
    resolved = resolve_path(path)
    if not resolved.exists():
        raise FileNotFoundError(f"Missing required {description}: {resolved}")
    return resolved

COLAB_DATA_CANDIDATES = [
    Path("/content/drive/MyDrive/industrial-surface-defect-inspection/data/raw/mvtec"),
    Path("/content/industrial-surface-defect-inspection/data/raw/mvtec"),
]

LOCAL_DATA_ROOT = REPO_ROOT / "data" / "raw" / "mvtec"
DATA_SOURCE_MODE = "unresolved/blocking"
DATA_ROOT = None

if IN_COLAB:
    for candidate in COLAB_DATA_CANDIDATES:
        if candidate.exists():
            DATA_ROOT = candidate.resolve()
            DATA_SOURCE_MODE = "Colab candidate"
            break
    if DATA_ROOT is None and LOCAL_DATA_ROOT.exists():
        DATA_ROOT = LOCAL_DATA_ROOT.resolve()
        DATA_SOURCE_MODE = "local fallback"
else:
    if LOCAL_DATA_ROOT.exists():
        DATA_ROOT = LOCAL_DATA_ROOT.resolve()
        DATA_SOURCE_MODE = "local fallback"

if DATA_ROOT is None or not DATA_ROOT.exists():
    print(f"data_source_mode={DATA_SOURCE_MODE}")
    raise FileNotFoundError("MVTec raw data root is required and was not resolved.")

RUN_CONFIG_PATH = resolve_required_path("configs/runs/mlp_train_supervised_v0_1_0.yaml", "MLP run config")
MODEL_CONFIG_PATH = resolve_required_path("configs/models/mlp_supervised.yaml", "MLP model config")
CNN_RUN_CONFIG_PATH = resolve_required_path("configs/runs/cnn_train_supervised_v0_1_0.yaml", "CNN run config")
CNN_MODEL_CONFIG_PATH = resolve_required_path("configs/models/cnn_supervised.yaml", "CNN model config")
MANIFEST_PATH = resolve_required_path("data/manifests/split_mvtec_classification_supervised.yaml", "supervised split manifest")
CLASS_MAPPING_PATH = resolve_required_path("configs/data/class_mapping_mvtec_binary.yaml", "class mapping")
COMPARISON_SCRIPT_PATH = resolve_required_path("scripts/evaluation/compare_track_a_classification.py", "Track A comparison script")

print(f"data_source_mode={DATA_SOURCE_MODE}")
print(f"data_root={DATA_ROOT}")
print(f"mlp_run_config={RUN_CONFIG_PATH}")
print(f"mlp_model_config={MODEL_CONFIG_PATH}")
print(f"cnn_run_config={CNN_RUN_CONFIG_PATH}")
print(f"cnn_model_config={CNN_MODEL_CONFIG_PATH}")
print(f"manifest={MANIFEST_PATH}")
print(f"class_mapping={CLASS_MAPPING_PATH}")
print(f"comparison_script={COMPARISON_SCRIPT_PATH}")


## 3. Config Summary

Load governed YAML configs and display key identifiers. The notebook reads these files; it does not redefine their values.

In [ ]:
import yaml

def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        payload = yaml.safe_load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"YAML must parse to a dictionary: {path}")
    return payload

run_config = load_yaml(RUN_CONFIG_PATH)
model_config = load_yaml(MODEL_CONFIG_PATH)
cnn_run_config = load_yaml(CNN_RUN_CONFIG_PATH)
cnn_model_config = load_yaml(CNN_MODEL_CONFIG_PATH)
manifest = load_yaml(MANIFEST_PATH)
class_mapping = load_yaml(CLASS_MAPPING_PATH)

summary = {
    "dataset_id": run_config["dataset_binding"]["dataset_id"],
    "manifest_dataset_id": manifest["dataset_id"],
    "dataset_version": run_config["dataset_binding"]["dataset_version"],
    "split_manifest_path": run_config["dataset_binding"]["split_manifest_path"],
    "class_mapping_path": run_config["dataset_binding"]["class_mapping_path"],
    "mlp_run_config_id": run_config["identity"]["run_config_id"],
    "mlp_model_config_id": run_config["model_identity"]["model_config_id"],
    "mlp_model_config_file_id": model_config["config_id"],
    "cnn_run_config_id": cnn_run_config["identity"]["run_config_id"],
    "cnn_model_config_id": cnn_run_config["model_identity"]["model_config_id"],
    "cnn_model_config_file_id": cnn_model_config["config_id"],
    "class_count": model_config["class_count"],
    "cnn_class_count": cnn_model_config["class_count"],
    "seed": run_config["training_runtime"]["seed"],
    "epochs": run_config["training_runtime"]["epochs"],
    "batch_size": run_config["training_runtime"]["batch_size"],
}

if summary["dataset_id"] != summary["manifest_dataset_id"]:
    raise ValueError("Run config dataset_id does not match manifest dataset_id.")
if cnn_run_config["dataset_binding"]["dataset_id"] != summary["dataset_id"]:
    raise ValueError("CNN run config dataset_id does not match MLP run config dataset_id.")
if cnn_run_config["dataset_binding"]["split_manifest_path"] != summary["split_manifest_path"]:
    raise ValueError("CNN run config split manifest does not match MLP Track A split manifest.")
if summary["mlp_model_config_id"] != summary["mlp_model_config_file_id"]:
    raise ValueError("MLP run config model_config_id does not match model config file config_id.")
if summary["cnn_model_config_id"] != summary["cnn_model_config_file_id"]:
    raise ValueError("CNN run config model_config_id does not match model config file config_id.")
if model_config["dataset_id"] != summary["dataset_id"]:
    raise ValueError("MLP model config dataset_id does not match run config dataset_id.")
if cnn_model_config["dataset_id"] != summary["dataset_id"]:
    raise ValueError("CNN model config dataset_id does not match run config dataset_id.")
if class_mapping["class_count"] != 2 or class_mapping["class_to_index"] != {"good": 0, "defect": 1}:
    raise ValueError("Class mapping must define binary good/defect classes.")
if summary["class_count"] != 2 or summary["cnn_class_count"] != 2:
    raise ValueError("Track A MLP and CNN configs must both define class_count=2.")

summary


## 4. Manifest Inspection

Inspect split counts, label distribution, category distribution, and overlap status from the governed manifest.

In [ ]:
from collections import Counter

SPLITS = ("train", "validation", "test")

def entries_for(split: str) -> list[dict]:
    entries = manifest.get(f"{split}_entries")
    if not isinstance(entries, list):
        raise ValueError(f"Manifest missing list field: {split}_entries")
    return entries

manifest_summary = {}
for split in SPLITS:
    entries = entries_for(split)
    manifest_summary[split] = {
        "count": len(entries),
        "labels": dict(sorted(Counter(entry["label"] for entry in entries).items())),
        "categories": dict(sorted(Counter(entry["category"] for entry in entries).items())),
    }

overlap_summary = {
    "overlap_check_status": manifest.get("overlap_check_status"),
    "train_validation_overlap_count": manifest.get("train_validation_overlap_count"),
    "train_test_overlap_count": manifest.get("train_test_overlap_count"),
    "validation_test_overlap_count": manifest.get("validation_test_overlap_count"),
}
if overlap_summary["overlap_check_status"] != "pass":
    raise ValueError("Manifest overlap check did not pass.")

{"manifest_summary": manifest_summary, "overlap_summary": overlap_summary}

## 5. Data Loader Smoke Inspection

Use the existing project loader from `src/inspection_ai/training/data_loading.py`. This cell performs a lightweight shape inspection only.

In [ ]:
from inspection_ai.training.data_loading import build_data_loaders


def require_loader_key(loaders: dict, key: str):
    if key not in loaders:
        raise KeyError(f"build_data_loaders output is missing required key: {key}")
    return loaders[key]


def safe_count_from_loader_output(loaders: dict, split: str) -> int:
    entries = loaders.get(split)
    if isinstance(entries, list):
        return len(entries)
    return len(entries_for(split))


data_loaders = build_data_loaders(run_config)
if not isinstance(data_loaders, dict):
    raise TypeError("build_data_loaders must return a dictionary for notebook inspection.")

loader_summary = {
    "available_keys": sorted(data_loaders.keys()),
    "train_count": safe_count_from_loader_output(data_loaders, "train"),
    "validation_count": safe_count_from_loader_output(data_loaders, "validation"),
    "test_count": safe_count_from_loader_output(data_loaders, "test"),
    "validation_loader_available": data_loaders.get("validation_loader") is not None,
}

train_loader = data_loaders.get("train_loader")
if train_loader is None:
    loader_summary["train_batch_shape_status"] = "train_loader unavailable"
else:
    first_train_batch = next(iter(train_loader))
    if not isinstance(first_train_batch, dict):
        raise TypeError("Expected train_loader batch to be a dictionary.")
    image_tensor = first_train_batch.get("image")
    label_tensor = first_train_batch.get("label")
    if image_tensor is not None and hasattr(image_tensor, "shape"):
        loader_summary["train_batch_image_shape"] = list(image_tensor.shape)
    else:
        loader_summary["train_batch_image_shape"] = "unavailable"
    if label_tensor is not None and hasattr(label_tensor, "shape"):
        loader_summary["train_batch_label_shape"] = list(label_tensor.shape)
    else:
        loader_summary["train_batch_label_shape"] = "unavailable"

loader_summary

## 6. Training Execution Placeholder

This notebook must call the existing governed training script. It must not implement a notebook-local training loop.

**Intentional execution gate:** set `RUN_TRAINING = True` only when you want this cell to run. Keep it `False` for documentation, inspection, and lightweight notebook review.

In [ ]:
import os
import re
import subprocess

RUN_TRAINING = False
training_stdout = None
training_result_path = None


def discover_latest_training_result() -> Path | None:
    artifact_dir = REPO_ROOT / "artifacts" / "models" / "analysis" / "training_results"
    if not artifact_dir.is_dir():
        return None
    candidates = sorted(
        artifact_dir.glob("training_result__*.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        print("Multiple TrainingResult artifacts found; latest fallback is intentionally not used because it is ambiguous.")
    return None


def parse_training_result_path(stdout: str) -> Path | None:
    match = re.search(r"Training result saved:\s*(.+)", stdout)
    if not match:
        return None
    return resolve_path(match.group(1).strip())

if RUN_TRAINING:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_PATH)
    command = [
        sys.executable,
        "scripts/training/train_model.py",
        "--config",
        str(RUN_CONFIG_PATH.relative_to(REPO_ROOT)),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, env=env, capture_output=True, text=True, check=False)
    training_stdout = completed.stdout
    print(training_stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Training command failed with exit code {completed.returncode}")
    training_result_path = parse_training_result_path(training_stdout)
    if training_result_path is None:
        print("TrainingResult path was not found in stdout; trying unambiguous latest-artifact fallback.")
        training_result_path = discover_latest_training_result()
    if training_result_path is None:
        raise RuntimeError("Unable to resolve TrainingResult path from stdout or unambiguous artifact discovery.")
else:
    print("Training execution skipped. Set RUN_TRAINING = True to call the governed training pipeline.")
    print("Stdout parsing and latest-artifact discovery are notebook orchestration only, not canonical artifact logic.")

## 7. Artifact Reading Placeholder

After training runs, read `TrainingResult` JSON and the linked `validation_evaluation_path`. This cell validates artifact structure for notebook display. It does not replace the source validation contract.

In [ ]:
import json

training_result = None
validation_evaluation = None
validation_evaluation_path = None


def resolve_artifact_path(path_value: str | Path) -> Path:
    if not isinstance(path_value, (str, Path)) or not str(path_value):
        raise ValueError("Artifact path must be a non-empty string or Path.")
    return resolve_path(path_value)

if training_result_path is None:
    print("No TrainingResult path available yet. Run the gated training cell first.")
else:
    training_result_path = resolve_artifact_path(training_result_path)
    if not training_result_path.is_file():
        raise FileNotFoundError(f"TrainingResult JSON not found: {training_result_path}")
    training_result = json.loads(training_result_path.read_text(encoding="utf-8"))
    validation_evaluation_path = resolve_artifact_path(training_result["metadata"]["validation_evaluation_path"])
    if not validation_evaluation_path.is_file():
        raise FileNotFoundError(f"Validation evaluation artifact not found: {validation_evaluation_path}")
    validation_evaluation = json.loads(validation_evaluation_path.read_text(encoding="utf-8"))
    cm = validation_evaluation["confusion_matrix"]
    if len(cm) != 2 or any(len(row) != 2 for row in cm):
        raise ValueError("Confusion matrix must be 2x2.")
    if sum(sum(row) for row in cm) != validation_evaluation["total_samples"]:
        raise ValueError("Confusion matrix sum must equal total_samples.")
    if validation_evaluation.get("run_id") != training_result["identity"].get("run_id"):
        raise ValueError("Evaluation artifact run_id does not match TrainingResult run_id.")
    if validation_evaluation.get("dataset_id") != training_result["metadata"].get("dataset_id"):
        raise ValueError("Evaluation artifact dataset_id does not match TrainingResult dataset_id.")
    print("Artifacts loaded and structurally valid for display.")

## 8. Visualization Placeholder

Visualize metrics and confusion matrix only when governed artifacts are available. Do not show fake values.

In [ ]:
if training_result is None or validation_evaluation is None:
    print("No artifacts available for visualization yet.")
else:
    metrics = training_result["metrics"]
    display_metrics = {
        "train_accuracy": metrics.get("train_accuracy"),
        "train_f1": metrics.get("train_f1"),
        "val_loss": metrics.get("val_loss"),
        "val_accuracy": metrics.get("val_accuracy"),
        "val_f1": metrics.get("val_f1"),
        "macro_precision": validation_evaluation["macro_metrics"].get("precision"),
        "macro_recall": validation_evaluation["macro_metrics"].get("recall"),
        "macro_f1": validation_evaluation["macro_metrics"].get("f1"),
    }
    print("Metric summary:")
    for key, value in display_metrics.items():
        print(f"  {key}: {value}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(validation_evaluation["confusion_matrix"])

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(4, 4))
        matrix = validation_evaluation["confusion_matrix"]
        image = ax.imshow(matrix, cmap="Blues")
        ax.set_xticks([0, 1], labels=["pred good", "pred defect"])
        ax.set_yticks([0, 1], labels=["true good", "true defect"])
        for row_index, row in enumerate(matrix):
            for col_index, value in enumerate(row):
                ax.text(col_index, row_index, str(value), ha="center", va="center", color="black")
        ax.set_title("Validation Confusion Matrix")
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib is unavailable; confusion matrix shown as text table only.")

## 9. Track A Comparison Execution

This section consumes governed `TrainingResult` artifacts from the MLP and CNN Track A runs and produces a comparison artifact by calling the existing project comparison script.

It exists to keep the notebook aligned with the current Track A decision system while preserving source-of-truth separation. The notebook does not compare models itself and does not implement decision logic.

Execution is gated. Missing MLP or CNN result paths are a blocking condition for comparison execution, but non-blocking for config, manifest, and data inspection.


In [ ]:
RUN_COMPARISON = False
comparison_stdout = None
comparison_artifact_path = None
comparison_artifact = None

# Set these explicitly after running both governed training commands, or assign
# them from known TrainingResult artifact paths. The notebook does not auto-select
# latest artifacts because that can create ambiguous evidence.
MLP_TRAINING_RESULT_PATH = training_result_path if training_result_path else None
CNN_TRAINING_RESULT_PATH = None


def parse_comparison_artifact_path(stdout: str) -> Path | None:
    match = re.search(r"comparison_artifact_path=(.+)", stdout)
    if not match:
        return None
    return resolve_artifact_path(match.group(1).strip())


def require_existing_artifact_path(path_value: str | Path | None, description: str) -> Path:
    if path_value is None:
        raise RuntimeError(f"Missing required artifact path: {description}")
    resolved = resolve_artifact_path(path_value)
    if not resolved.is_file():
        raise FileNotFoundError(f"{description} not found: {resolved}")
    return resolved

if RUN_COMPARISON:
    mlp_result_for_comparison = require_existing_artifact_path(MLP_TRAINING_RESULT_PATH, "MLP TrainingResult")
    cnn_result_for_comparison = require_existing_artifact_path(CNN_TRAINING_RESULT_PATH, "CNN TrainingResult")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_PATH)
    command = [
        sys.executable,
        str(COMPARISON_SCRIPT_PATH.relative_to(REPO_ROOT)),
        "--mlp-result",
        str(mlp_result_for_comparison.relative_to(REPO_ROOT)),
        "--cnn-result",
        str(cnn_result_for_comparison.relative_to(REPO_ROOT)),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, env=env, capture_output=True, text=True, check=False)
    comparison_stdout = completed.stdout
    print(comparison_stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Track A comparison command failed with exit code {completed.returncode}")
    comparison_artifact_path = parse_comparison_artifact_path(comparison_stdout)
    if comparison_artifact_path is None:
        raise RuntimeError("Comparison artifact path was not found in comparison stdout.")
else:
    print("Comparison execution skipped. Set RUN_COMPARISON = True after MLP and CNN TrainingResult paths are available.")
    print("Required inputs: MLP_TRAINING_RESULT_PATH and CNN_TRAINING_RESULT_PATH.")
    print("Canonical decision logic remains in src/inspection_ai/evaluation/comparison.py and the comparison script.")


## 10. Comparison Artifact Inspection

This section consumes a comparison JSON artifact from `artifacts/models/comparisons/` and presents the governed decision payload for review.

If no comparison artifact path is available, this section reports the blocker. It does not fabricate comparison metrics or decision outputs.


In [ ]:
comparison_artifact = None
candidate_rows = []

if comparison_artifact_path is None:
    print("No comparison artifact path available. Run the gated comparison section or set comparison_artifact_path explicitly.")
else:
    comparison_artifact_path = resolve_artifact_path(comparison_artifact_path)
    if not comparison_artifact_path.is_file():
        raise FileNotFoundError(f"Comparison artifact not found: {comparison_artifact_path}")
    comparison_artifact = json.loads(comparison_artifact_path.read_text(encoding="utf-8"))
    if comparison_artifact.get("dataset_id") != summary["dataset_id"]:
        raise ValueError("Comparison artifact dataset_id does not match Track A dataset_id.")
    policy = comparison_artifact.get("decision_policy")
    if not isinstance(policy, dict):
        raise ValueError("Comparison artifact is missing decision_policy.")
    candidates = comparison_artifact.get("candidates")
    if not isinstance(candidates, list) or len(candidates) != 2:
        raise ValueError("Comparison artifact must contain exactly two candidates.")
    recommended_candidate = comparison_artifact.get("recommended_candidate")
    if not isinstance(recommended_candidate, dict):
        raise ValueError("Comparison artifact is missing recommended_candidate.")

    required_candidate_fields = {
        "model_name",
        "macro_f1",
        "defect_recall",
        "defect_precision",
        "false_negatives",
        "warnings",
        "explanation",
    }
    for candidate in candidates:
        missing = sorted(required_candidate_fields - set(candidate))
        if missing:
            raise ValueError(f"Comparison candidate is missing fields: {missing}")
        candidate_rows.append({
            "model_name": candidate["model_name"],
            "macro_f1": candidate["macro_f1"],
            "defect_recall": candidate["defect_recall"],
            "defect_precision": candidate["defect_precision"],
            "false_negatives": candidate["false_negatives"],
            "recommendation_status": candidate.get("recommendation_status"),
        })

    print("decision_policy:")
    print(json.dumps(policy, indent=2))
    print("candidate_table:")
    for row in candidate_rows:
        print(row)
    print("candidate_warnings_and_explanations:")
    for candidate in candidates:
        print(f"{candidate['model_name']} warnings={candidate['warnings']}")
        print(f"{candidate['model_name']} explanation={candidate['explanation']}")
    print("recommended_candidate:")
    print(json.dumps(recommended_candidate, indent=2))


## 11. Comparison Visualization Placeholder

This section displays the comparison artifact in a reviewer-friendly table when the artifact exists. It reads artifact fields only; it does not recompute metrics or decisions.

Plotting is optional. If display helpers are unavailable, text output from the previous section remains the fallback.


In [ ]:
if comparison_artifact is None:
    print("No comparison artifact available for table display.")
else:
    try:
        import pandas as pd
    except ImportError:
        print("pandas is unavailable; candidate table shown as plain Python dictionaries.")
        for row in candidate_rows:
            print(row)
    else:
        display(pd.DataFrame(candidate_rows))


## 12. Sample Input Inspection

Show sample input images and true labels only when explicitly enabled. This section does not show predicted labels unless a governed prediction artifact exists.


In [ ]:
SHOW_SAMPLE_IMAGES = False


def resolve_entry_path(entry_path: str | Path) -> Path:
    return resolve_path(entry_path)

if not SHOW_SAMPLE_IMAGES:
    print("Sample image display skipped. Set SHOW_SAMPLE_IMAGES = True for qualitative input inspection.")
else:
    try:
        from PIL import Image
        import matplotlib.pyplot as plt
    except ImportError:
        print("PIL or matplotlib is unavailable; sample image display skipped.")
    else:
        sample_entries = entries_for("validation")[:8]
        loaded_samples = []
        for entry in sample_entries:
            image_path = resolve_entry_path(entry["path"])
            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as exc:
                print(f"Unable to load sample image {image_path}: {exc}")
                continue
            loaded_samples.append((entry, image))

        if not loaded_samples:
            print("No sample images could be loaded.")
        else:
            fig, axes = plt.subplots(2, 4, figsize=(12, 6))
            for axis in axes.ravel():
                axis.axis("off")
            for axis, (entry, image) in zip(axes.ravel(), loaded_samples):
                axis.imshow(image)
                axis.set_title(f"{entry['category']} / true={entry['label']}", fontsize=9)
                axis.axis("off")
            plt.tight_layout()
            print("Predicted labels are intentionally not shown unless a governed prediction artifact exists.")

## 13. Explainability Placeholder

Explainability artifacts are not implemented yet for Track A. This notebook must not generate fake explainability output.


## 14. Output And Decision Summary

Summarize config identity, dataset identity, consumed artifacts, produced or loaded comparison artifact, recommended candidate, review status, decision rationale, and next project step when artifacts are available.


In [ ]:
output_summary = {
    "dataset_id": manifest["dataset_id"],
    "split_version": manifest.get("split_version"),
    "split_seed": manifest.get("split_seed"),
    "mlp_run_config_id": run_config["identity"]["run_config_id"],
    "cnn_run_config_id": cnn_run_config["identity"]["run_config_id"],
    "mlp_model_config_id": model_config["config_id"],
    "cnn_model_config_id": cnn_model_config["config_id"],
    "training_result_path": str(training_result_path) if training_result_path else None,
    "validation_evaluation_path": str(validation_evaluation_path) if validation_evaluation_path else None,
    "comparison_artifact_path": str(comparison_artifact_path) if comparison_artifact_path else None,
}
if training_result is not None:
    output_summary["run_id"] = training_result["identity"]["run_id"]
    output_summary["metrics"] = {key: training_result["metrics"].get(key) for key in ("train_accuracy", "train_f1", "val_loss", "val_accuracy", "val_f1")}
if comparison_artifact is not None:
    recommended = comparison_artifact["recommended_candidate"]
    output_summary["recommended_model_type"] = recommended.get("model_type")
    output_summary["recommendation_status"] = recommended.get("recommendation_status")
    output_summary["decision_explanation"] = recommended.get("decision_explanation")
    output_summary["consumed_training_results"] = [candidate.get("training_result_path") for candidate in comparison_artifact["candidates"]]
    output_summary["consumed_validation_evaluations"] = [candidate.get("validation_evaluation_path") for candidate in comparison_artifact["candidates"]]
    output_summary["next_project_step"] = "Promote Track A decision thresholds to governed config after review."
else:
    output_summary["comparison_status"] = "not_available"
    output_summary["comparison_blocker"] = "Run gated comparison section with explicit MLP and CNN TrainingResult paths."
output_summary


## 15. Final Decision Summary

Use this section after executing the gated cells to summarize:

- Whether config and manifest checks passed.
- Whether data loader smoke inspection passed.
- Which TrainingResult and validation evaluation artifacts were consumed.
- Which Track A comparison artifact was produced or loaded.
- Which candidate was recommended.
- Whether `review_required` was triggered and why.
- Current limitations: lightweight one-epoch/one-batch training behavior, bounded validation artifact evaluation, comparison based on current policy thresholds, and supervised split is not the official MVTec anomaly benchmark.
- Next project step: promote decision thresholds to governed config and extend evidence only after review.
